In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install transformers datasets torch scikit-learn

In [3]:
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [4]:
import torch
print("GPU Available:", torch.cuda.is_available())

GPU Available: True


In [5]:
import os

base_path = "/content/drive/MyDrive/Aurevia/data/raw/GoEmotions/"

print(os.listdir(base_path))

['goemotions_1.csv', 'goemotions_2.csv', 'goemotions_3.csv']


In [6]:
import pandas as pd

df1 = pd.read_csv(base_path + "goemotions_1.csv")
df2 = pd.read_csv(base_path + "goemotions_2.csv")
df3 = pd.read_csv(base_path + "goemotions_3.csv")

print("Shapes:")
print(df1.shape, df2.shape, df3.shape)

Shapes:
(70000, 37) (70000, 37) (71225, 37)


In [7]:
df = pd.concat([df1, df2, df3], ignore_index=True)

print("Final Shape:", df.shape)
df.head()

Final Shape: (211225, 37)


,text,id,author,subreddit,link_id,parent_id,created_utc,rater_id,example_very_unclear,admiration,...,love,nervousness,optimism,pride,realization,relief,remorse,sadness,surprise,neutral
0,That game hurt.,eew5j0j,Brdd9,nrl,t3_ajis4z,t1_eew18eq,1.548381e+09,1,False,0,...,0,0,0,0,0,0,0,1,0,0
1,>sexuality shouldn’t be a grouping category I...,eemcysk,TheGreen888,unpopularopinion,t3_ai4q37,t3_ai4q37,1.548084e+09,37,True,0,...,0,0,0,0,0,0,0,0,0,0
2,"You do right, if you don't care then fuck 'em!",ed2mah1,Labalool,confessions,t3_abru74,t1_ed2m7g7,1.546428e+09,37,False,0,...,0,0,0,0,0,0,0,0,0,1
3,Man I love reddit.,eeibobj,MrsRobertshaw,facepalm,t3_ahulml,t3_ahulml,1.547965e+09,18,False,0,...,1,0,0,0,0,0,0,0,0,0
4,"[NAME] was nowhere near them, he was by the Fa...",eda6yn6,American_Fascist713,starwarsspeculation,t3_ackt2f,t1_eda65q2,1.546669e+09,2,False,0,...,0,0,0,0,0,0,0,0,0,1


In [8]:
print(df.columns)

Index(['text', 'id', 'author', 'subreddit', 'link_id', 'parent_id',
       'created_utc', 'rater_id', 'example_very_unclear', 'admiration',
       'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion',
       'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust',
       'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy',
       'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief',
       'remorse', 'sadness', 'surprise', 'neutral'],
      dtype='object')


In [9]:
print(df.columns)
df.head()

Index(['text', 'id', 'author', 'subreddit', 'link_id', 'parent_id',
       'created_utc', 'rater_id', 'example_very_unclear', 'admiration',
       'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion',
       'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust',
       'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy',
       'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief',
       'remorse', 'sadness', 'surprise', 'neutral'],
      dtype='object')


,text,id,author,subreddit,link_id,parent_id,created_utc,rater_id,example_very_unclear,admiration,...,love,nervousness,optimism,pride,realization,relief,remorse,sadness,surprise,neutral
0,That game hurt.,eew5j0j,Brdd9,nrl,t3_ajis4z,t1_eew18eq,1.548381e+09,1,False,0,...,0,0,0,0,0,0,0,1,0,0
1,>sexuality shouldn’t be a grouping category I...,eemcysk,TheGreen888,unpopularopinion,t3_ai4q37,t3_ai4q37,1.548084e+09,37,True,0,...,0,0,0,0,0,0,0,0,0,0
2,"You do right, if you don't care then fuck 'em!",ed2mah1,Labalool,confessions,t3_abru74,t1_ed2m7g7,1.546428e+09,37,False,0,...,0,0,0,0,0,0,0,0,0,1
3,Man I love reddit.,eeibobj,MrsRobertshaw,facepalm,t3_ahulml,t3_ahulml,1.547965e+09,18,False,0,...,1,0,0,0,0,0,0,0,0,0
4,"[NAME] was nowhere near them, he was by the Fa...",eda6yn6,American_Fascist713,starwarsspeculation,t3_ackt2f,t1_eda65q2,1.546669e+09,2,False,0,...,0,0,0,0,0,0,0,0,0,1


In [10]:
burnout_emotions = [
    'anger', 'annoyance', 'disappointment', 
    'disapproval', 'grief', 'nervousness',
    'remorse', 'sadness'
]

df['burnout'] = df[burnout_emotions].max(axis=1)

df[['text','burnout']].head()

,text,burnout
0,That game hurt.,1
1,>sexuality shouldn’t be a grouping category I...,0
2,"You do right, if you don't care then fuck 'em!",0
3,Man I love reddit.,0
4,"[NAME] was nowhere near them, he was by the Fa...",0


In [11]:
df['burnout'].value_counts()

,count
burnout,
0,164055
1,47170


In [12]:
df_clean = df[['text', 'burnout']].copy()

print(df_clean.shape)
df_clean.head()

(211225, 2)


,text,burnout
0,That game hurt.,1
1,>sexuality shouldn’t be a grouping category I...,0
2,"You do right, if you don't care then fuck 'em!",0
3,Man I love reddit.,0
4,"[NAME] was nowhere near them, he was by the Fa...",0


In [13]:
df_clean = df_clean.dropna()
df_clean = df_clean[df_clean['text'].str.len() > 5]

print("After cleaning:", df_clean.shape)

After cleaning: (211090, 2)


In [14]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df_clean,
    test_size=0.2,
    stratify=df_clean['burnout'],
    random_state=42
)

print("Train:", train_df.shape)
print("Test:", test_df.shape)

Train: (168872, 2)
Test: (42218, 2)


In [15]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

print(train_dataset)
print(test_dataset)

Dataset({
    features: ['text', 'burnout'],
    num_rows: 168872
})
Dataset({
    features: ['text', 'burnout'],
    num_rows: 42218
})


In [16]:
from transformers import AutoTokenizer

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [17]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/168872 [00:00<?, ? examples/s]

Map:   0%|          | 0/42218 [00:00<?, ? examples/s]

In [18]:
train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "burnout"]
)

test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "burnout"]
)

In [19]:
train_dataset = train_dataset.rename_column("burnout", "labels")
test_dataset = test_dataset.rename_column("burnout", "labels")

In [20]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [21]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='binary'
    )
    
    acc = accuracy_score(labels, predictions)
    
    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

In [22]:
!pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 112.8 MB/s eta 0:00:0000:01:01
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [23]:
import transformers
print(transformers.__version__)

5.0.0


In [24]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/Aurevia/results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_dir="/content/drive/MyDrive/Aurevia/logs"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [25]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [26]:
trainer.evaluate()

{'eval_loss': 0.5888205170631409,
 'eval_model_preparation_time': 0.0028,
 'eval_accuracy': 0.7764460656591975,
 'eval_f1': 0.0019035532994923859,
 'eval_precision': 0.34615384615384615,
 'eval_recall': 0.0009544008483563097,
 'eval_runtime': 356.008,
 'eval_samples_per_second': 118.587,
 'eval_steps_per_second': 7.413}

In [27]:
trainer.save_model("aurevia_text_model_stage_one")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Third Time Training

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd
import os

base_path = "/content/drive/MyDrive/Aurevia/data/raw/GoEmotions/"

df1 = pd.read_csv(base_path + "goemotions_1.csv")
df2 = pd.read_csv(base_path + "goemotions_2.csv")
df3 = pd.read_csv(base_path + "goemotions_3.csv")

df = pd.concat([df1, df2, df3], ignore_index=True)

print("Total shape:", df.shape)
df.head()

Total shape: (211225, 37)


,text,id,author,subreddit,link_id,parent_id,created_utc,rater_id,example_very_unclear,admiration,...,love,nervousness,optimism,pride,realization,relief,remorse,sadness,surprise,neutral
0,That game hurt.,eew5j0j,Brdd9,nrl,t3_ajis4z,t1_eew18eq,1.548381e+09,1,False,0,...,0,0,0,0,0,0,0,1,0,0
1,>sexuality shouldn’t be a grouping category I...,eemcysk,TheGreen888,unpopularopinion,t3_ai4q37,t3_ai4q37,1.548084e+09,37,True,0,...,0,0,0,0,0,0,0,0,0,0
2,"You do right, if you don't care then fuck 'em!",ed2mah1,Labalool,confessions,t3_abru74,t1_ed2m7g7,1.546428e+09,37,False,0,...,0,0,0,0,0,0,0,0,0,1
3,Man I love reddit.,eeibobj,MrsRobertshaw,facepalm,t3_ahulml,t3_ahulml,1.547965e+09,18,False,0,...,1,0,0,0,0,0,0,0,0,0
4,"[NAME] was nowhere near them, he was by the Fa...",eda6yn6,American_Fascist713,starwarsspeculation,t3_ackt2f,t1_eda65q2,1.546669e+09,2,False,0,...,0,0,0,0,0,0,0,0,0,1


In [3]:
df.columns

Index(['text', 'id', 'author', 'subreddit', 'link_id', 'parent_id',
       'created_utc', 'rater_id', 'example_very_unclear', 'admiration',
       'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion',
       'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust',
       'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy',
       'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief',
       'remorse', 'sadness', 'surprise', 'neutral'],
      dtype='object')

In [4]:
burnout_emotions = [
    'anger', 'annoyance', 'disappointment',
    'disapproval', 'disgust', 'fear',
    'grief', 'nervousness', 'remorse',
    'sadness'
]

def convert_to_binary(row):
    for emotion in burnout_emotions:
        if row[emotion] == 1:
            return 1
    return 0

df["binary_label"] = df.apply(convert_to_binary, axis=1)

df["binary_label"].value_counts()

,count
binary_label,
0,158157
1,53068


In [5]:
from sklearn.utils import resample

majority = df[df.binary_label == 0]
minority = df[df.binary_label == 1]

majority_downsampled = resample(
    majority,
    replace=False,
    n_samples=len(minority),
    random_state=42
)

df_balanced = pd.concat([majority_downsampled, minority])

df_balanced["binary_label"].value_counts()

,count
binary_label,
0,53068
1,53068


In [6]:
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df_balanced["text"],
    df_balanced["binary_label"],
    test_size=0.2,
    random_state=42,
    stratify=df_balanced["binary_label"]
)

In [7]:
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df_balanced["text"],
    df_balanced["binary_label"],
    test_size=0.2,
    random_state=42,
    stratify=df_balanced["binary_label"]
)

print("Train size:", len(train_texts))
print("Validation size:", len(val_texts))

Train size: 84908
Validation size: 21228


In [8]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


In [10]:
train_encodings = tokenizer(
    list(train_texts),
    truncation=True,
    padding=True,
    max_length=128
)

val_encodings = tokenizer(
    list(val_texts),
    truncation=True,
    padding=True,
    max_length=128
)

In [11]:
import torch

class BurnoutDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.reset_index(drop=True)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels.iloc[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = BurnoutDataset(train_encodings, train_labels)
val_dataset = BurnoutDataset(val_encodings, val_labels)

In [12]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Fourth Time - Laptop Training (Full Bias)

In [1]:
pip install torch transformers scikit-learn pandas wandb

Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: torch in c:\users\thimathi\appdata\local\programs\python\python310\lib\site-packages (2.10.0)
     ---------------------------------------- 24.6/24.6 MB 2.3 MB/s eta 0:00:00
     -------------------------------------- 208.6/208.6 kB 1.4 MB/s eta 0:00:00
     -------------------------------------- 437.1/437.1 kB 1.6 MB/s eta 0:00:00
     -------------------------------------- 463.6/463.6 kB 1.2 MB/s eta 0:00:00
     -------------------------------------- 437.9/437.9 kB 2.0 MB/s eta 0:00:00
     ---------------------------------------- 62.8/62.8 kB 3.5 MB/s eta 0:00:00
     ---------------------------------------- 2.0/2.0 MB 2.2 MB/s eta 0:00:00




[notice] A new release of pip available: 22.2.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
!wandb login

wandb: ERROR Find detailed error logs at: C:\Users\Thimathi\AppData\Local\Temp\debug-cli.Thimathi.log
Error: No API key configured. Use `wandb login` to log in.


In [8]:
pip install wandb

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.2.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
import wandb

info = wandb.login()
print(info)

True


In [17]:
import random
import os
import wandb

# -----------------------
# AUTO LOGIN
# -----------------------
wandb.login(key="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")

# -----------------------
# START RUN
# -----------------------
run = wandb.init(
    entity="chathuradissanayake274-chathura",
    project="2nd",
    config={
        "learning_rate": 0.02,
        "architecture": "CNN",
        "dataset": "CIFAR-100",
        "epochs": 10,
    },
)

# -----------------------
# SIMULATED TRAINING
# -----------------------
epochs = 10
offset = random.random() / 5

for epoch in range(2, epochs):
    acc = 1 - 2**-epoch - random.random() / epoch - offset
    loss = 2**-epoch + random.random() / epoch + offset

    run.log({
        "epoch": epoch,
        "accuracy": acc,
        "loss": loss
    })

run.finish()

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\Thimathi\_netrc
wandb: Currently logged in as: chathuradissanayake274 (chathuradissanayake274-chathura) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


accuracy,▁▁▁▄▇▇▇█
epoch,▁▂▃▄▅▆▇█
loss,█▆▃▂▁▁▁▁
accuracy,0.9863
epoch,9
loss,0.01935


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch
import pandas as pd
import numpy as np
import wandb
import accelerate
print(accelerate.__version__)
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer


# -----------------------
# W&B INIT
# -----------------------
wandb.init(project="Aurevia_Burnout_Detection", name="bert_binary_v1")

# -----------------------
# LOAD DATA
# -----------------------

import pandas as pd

df = pd.read_csv(r"C:\Users\Thimathi\source\Aurevia\data\raw\GoEmotions\goemotions_1.csv")
df2 = pd.read_csv(r"C:\Users\Thimathi\source\Aurevia\data\raw\GoEmotions\goemotions_2.csv")
df3 = pd.read_csv(r"C:\Users\Thimathi\source\Aurevia\data\raw\GoEmotions\goemotions_3.csv")

df = pd.concat([df, df2, df3], ignore_index=True)

# -----------------------
# DEFINE BURNOUT EMOTIONS
# -----------------------
burnout_emotions = [
    'anger', 'annoyance', 'disappointment',
    'disapproval', 'disgust', 'fear',
    'grief', 'nervousness', 'remorse',
    'sadness'
]

def convert_to_binary(row):
    for emotion in burnout_emotions:
        if row[emotion] == 1:
            return 1
    return 0

df["binary_label"] = df.apply(convert_to_binary, axis=1)

# -----------------------
# BALANCE DATASET (Downsample)
# -----------------------
majority = df[df.binary_label == 0]
minority = df[df.binary_label == 1]

majority_downsampled = resample(
    majority,
    replace=False,
    n_samples=len(minority),
    random_state=42
)

sample_size = 20000

majority_sample = majority.sample(n=sample_size, random_state=42)
minority_sample = minority.sample(n=sample_size, random_state=42)

df_balanced = pd.concat([majority_sample, minority_sample])

# -----------------------
# TRAIN TEST SPLIT
# -----------------------
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df_balanced["text"],
    df_balanced["binary_label"],
    test_size=0.2,
    random_state=42,
    stratify=df_balanced["binary_label"]
)

# -----------------------
# TOKENIZER
# -----------------------
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

train_encodings = tokenizer(
    list(train_texts),
    truncation=True,
    padding=True,
    max_length=128
)

val_encodings = tokenizer(
    list(val_texts),
    truncation=True,
    padding=True,
    max_length=128
)

class BurnoutDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.reset_index(drop=True)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels.iloc[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = BurnoutDataset(train_encodings, train_labels)
val_dataset = BurnoutDataset(val_encodings, val_labels)

# -----------------------
# MODEL
# -----------------------
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

# -----------------------
# METRICS
# -----------------------
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

# -----------------------
# TRAINING CONFIG
# -----------------------
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",      # ← IMPORTANT
    save_strategy="epoch",            # ← MUST MATCH
    logging_strategy="steps",
    logging_steps=200,
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    report_to="wandb",
    save_total_limit=2
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

trainer.save_model("./best_model")

wandb.finish()

1.12.0


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


train/epoch,▁█
train/global_step,▁█
train/grad_norm,▁█
train/learning_rate,█▁
train/loss,█▁
train/epoch,0.07537
train/global_step,400
train/grad_norm,6.15481
train/learning_rate,2e-05
train/loss,0.53089


Loading weights: 100%|██████████| 199/199 [00:01<00:00, 198.00it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those pa

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.486248,0.461917,0.779875,0.796956,0.739568,0.864000
2,0.406582,0.474004,0.781875,0.789987,0.761662,0.820500


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.32it/s]
c:\Users\Thimathi\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.28it/s]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'ber

eval/accuracy,▁█
eval/f1,█▁
eval/loss,▁█
eval/precision,▁█
eval/recall,█▁
eval/runtime,█▁
eval/samples_per_second,▁█
eval/steps_per_second,▁█
train/epoch,▁▂▃▃▄▄▅▆▆▇███
train/global_step,▁▂▃▃▄▄▅▆▆▇███
+3,...


In [4]:
import torch
import pandas as pd
import numpy as np
import wandb
import accelerate
print(accelerate.__version__)
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer


# -----------------------
# W&B INIT
# -----------------------
wandb.init(project="Aurevia_Burnout_Detection", name="bert_binary_v1")

# -----------------------
# LOAD DATA
# -----------------------

import pandas as pd

df = pd.read_csv(r"C:\Users\Thimathi\source\Aurevia\data\raw\GoEmotions\goemotions_1.csv")
df2 = pd.read_csv(r"C:\Users\Thimathi\source\Aurevia\data\raw\GoEmotions\goemotions_2.csv")
df3 = pd.read_csv(r"C:\Users\Thimathi\source\Aurevia\data\raw\GoEmotions\goemotions_3.csv")

df = pd.concat([df, df2, df3], ignore_index=True)

# -----------------------
# DEFINE BURNOUT EMOTIONS
# -----------------------
burnout_emotions = [
    'anger', 'annoyance', 'disappointment',
    'disapproval', 'disgust', 'fear',
    'grief', 'nervousness', 'remorse',
    'sadness'
]

def convert_to_binary(row):
    for emotion in burnout_emotions:
        if row[emotion] == 1:
            return 1
    return 0

df["binary_label"] = df.apply(convert_to_binary, axis=1)

# -----------------------
# BALANCE DATASET (Downsample)
# -----------------------
majority = df[df.binary_label == 0]
minority = df[df.binary_label == 1]

majority_downsampled = resample(
    majority,
    replace=False,
    n_samples=len(minority),
    random_state=42
)

sample_size = 20000

majority_sample = majority.sample(n=sample_size, random_state=42)
minority_sample = minority.sample(n=sample_size, random_state=42)

df_balanced = pd.concat([majority_sample, minority_sample])

# -----------------------
# TRAIN TEST SPLIT
# -----------------------
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df_balanced["text"],
    df_balanced["binary_label"],
    test_size=0.2,
    random_state=42,
    stratify=df_balanced["binary_label"]
)

# -----------------------
# TOKENIZER
# -----------------------
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

train_encodings = tokenizer(
    list(train_texts),
    truncation=True,
    padding=True,
    max_length=128
)

val_encodings = tokenizer(
    list(val_texts),
    truncation=True,
    padding=True,
    max_length=128
)

class BurnoutDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.reset_index(drop=True)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels.iloc[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = BurnoutDataset(train_encodings, train_labels)
val_dataset = BurnoutDataset(val_encodings, val_labels)

# -----------------------
# MODEL
# -----------------------
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

# -----------------------
# METRICS
# -----------------------
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

# -----------------------
# TRAINING CONFIG
# -----------------------
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",      # ← IMPORTANT
    save_strategy="epoch",            # ← MUST MATCH
    logging_strategy="steps",
    logging_steps=200,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    report_to="wandb",
    save_total_limit=2
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

trainer.save_model("./best_model_2")

wandb.finish()

1.12.0


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


train/epoch,▁█
train/global_step,▁█
train/grad_norm,▁█
train/learning_rate,█▁
train/loss,█▁
train/epoch,0.1
train/global_step,400
train/grad_norm,4.84949
train/learning_rate,2e-05
train/loss,0.51943


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 206.32it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those pa

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.489564,0.465185,0.777375,0.795921,0.734715,0.868250
2,0.407387,0.488069,0.774125,0.779284,0.761882,0.797500
3,0.325153,0.555566,0.771625,0.779161,0.754271,0.805750


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.30it/s]
c:\Users\Thimathi\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.20it/s]
c:\Users\Thimathi\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.12it/s]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNor

eval/accuracy,█▄▁
eval/f1,█▁▁
eval/loss,▁▃█
eval/precision,▁█▆
eval/recall,█▁▂
eval/runtime,▇▁█
eval/samples_per_second,▂█▁
eval/steps_per_second,▂█▁
train/epoch,▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇█████
train/global_step,▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇█████
+3,...
